# Wrangle a single-cell cohort

This notebook runs a complete wrangling pass over Scanpy's PBMC3K: 2,638
peripheral blood mononuclear cells with Louvain cell-type labels, QC metrics,
PCA/UMAP embeddings, and log-normalised counts for 13,714 genes in `.raw`.

Nothing here is simulated except two columns, which are marked where they
appear.

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata",
#     "annplyr",
#     "numpy",
#     "pandas",
#     "scanpy",
# ]
# ///

## Load the data

In [2]:
import pandas as pd
import scanpy as sc

import annplyr as ap
from annplyr.data import X, obs, raw

adata = ap.tidy(sc.datasets.pbmc3k_processed())
adata

AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'
    layers: None

`X` holds scaled values, so the interpretable log-normalised counts come from
`raw`. Each name below says which container it reads.

In [3]:
adata.count(obs.louvain, sort=True)

,louvain,n
0,CD4 T cells,1144
1,CD14+ Monocytes,480
2,B cells,342
3,CD8 T cells,316
4,NK cells,154
5,FCGR3A+ Monocytes,150
6,Dendritic cells,37
7,Megakaryocytes,15


## Attach donor metadata

PBMC3K is a single donor. The next cell assigns three synthetic donor labels so
the sample-aware verbs have something to work with. `donor_id` and `condition`
are the only invented values in this notebook.

In [4]:
adata.obs["donor_id"] = [f"donor_{i % 3 + 1}" for i in range(adata.n_obs)]

sample_sheet = pd.DataFrame(
    {
        "donor_id": ["donor_1", "donor_2", "donor_3"],
        "condition": ["control", "stimulated", "stimulated"],
        "collection_day": [0, 7, 7],
    }
)

cohort = adata.left_join(sample_sheet, by="donor_id", relationship="many-to-one")
cohort.obs[["louvain", "donor_id", "condition", "collection_day"]].head()

,louvain,donor_id,condition,collection_day
index,,,,
AAACATACAACCAC-1,CD4 T cells,donor_1,control,0
AAACATTGAGCTAC-1,B cells,donor_2,stimulated,7
AAACATTGATCAGC-1,CD4 T cells,donor_3,stimulated,7
AAACCGTGCTTCCG-1,CD14+ Monocytes,donor_1,control,0
AAACCGTGTATGCG-1,NK cells,donor_2,stimulated,7


`relationship="many-to-one"` is a check, not a comment: a duplicated donor row
would raise instead of silently duplicating cells that have no matching rows in
`X`.

## Apply the QC rule

In [5]:
qc = cohort.filter(
    obs.n_genes >= 500,
    obs.percent_mito < 0.03,
)

qc.shape

(2117, 1838)

## Derive a marker score

Matrix containers are read-only sources. This reads two genes out of 13,714 and
writes one `obs` column.

In [6]:
scored = qc.mutate(
    b_score=(raw.MS4A1 + raw.CD79A) / 2,
    max_matrix_values=2 * qc.n_obs,
)

scored.obs[["louvain", "donor_id", "b_score"]].head()

,louvain,donor_id,b_score
index,,,
AAACATTGATCAGC-1,CD4 T cells,donor_3,0.000000
AAACCGTGCTTCCG-1,CD14+ Monocytes,donor_1,0.000000
AAACCGTGTATGCG-1,NK cells,donor_2,0.000000
AAACGCACTGGTAC-1,CD8 T cells,donor_3,0.000000
AAACGCTGTAGCCA-1,CD4 T cells,donor_3,0.346574


## Rank within donor, then summarize

In [7]:
ranked = scored.group_by(obs.donor_id).mutate(depth_rank=ap.min_rank(obs.n_counts, descending=True))

ranked.filter(obs.depth_rank <= 2).ungroup().obs[["donor_id", "n_counts", "depth_rank"]]

,donor_id,n_counts,depth_rank
index,,,
CAGGTTGAGGATCT-1,donor_3,8011.0,1.0
CGATCAGATGTGAC-1,donor_3,6908.0,2.0
ACGAACTGGCTATG-1,donor_1,8875.0,1.0
CATACTTGGGTTAC-1,donor_1,7167.0,2.0
ACGAGGGACAGGAG-1,donor_2,7928.0,2.0
GGGCCAACCTTGGA-1,donor_2,8415.0,1.0


In [8]:
ranked.ungroup().summarize(
    cells=ap.n(),
    mean_b_score=ap.mean(obs.b_score),
    mean_LYZ=ap.mean(raw.LYZ),
    by=[obs.condition, obs.louvain],
).round(3)

,condition,louvain,cells,mean_b_score,mean_LYZ
0,control,CD4 T cells,337,0.043,0.462
1,control,CD14+ Monocytes,118,0.029,3.733
2,control,B cells,84,1.286,0.355
3,control,CD8 T cells,70,0.015,0.310
4,control,NK cells,46,0.045,0.376
5,control,FCGR3A+ Monocytes,44,0.039,2.108
6,control,Dendritic cells,10,0.035,4.103
7,control,Megakaryocytes,2,0.173,1.546
8,stimulated,CD4 T cells,648,0.030,0.442
9,stimulated,CD14+ Monocytes,231,0.038,3.755


The B-cell score and monocyte `LYZ` track cell type rather than condition,
which is what should happen when the condition labels are arbitrary.

## Produce the analysis object and the plotting table

In [9]:
analysis = ranked.ungroup().select(
    obs.donor_id,
    obs.condition,
    obs.louvain,
    obs.n_counts,
    obs.b_score,
    obs.depth_rank,
    X.MS4A1,
    X.CD79A,
    X.NKG7,
)

plot_data = analysis.join_features(
    raw.MS4A1,
    raw.CD79A,
    raw.NKG7,
    shape="long",
    max_matrix_values=3 * analysis.n_obs,
)

print(analysis.shape, plot_data.shape)
plot_data.head()

(2117, 3) (6351, 9)


,.cell,.feature,.abundance_raw,donor_id,condition,louvain,n_counts,b_score,depth_rank
0,AAACATTGATCAGC-1,MS4A1,0.0,donor_3,stimulated,CD4 T cells,3147.0,0.0,125.0
1,AAACATTGATCAGC-1,CD79A,0.0,donor_3,stimulated,CD4 T cells,3147.0,0.0,125.0
2,AAACATTGATCAGC-1,NKG7,0.0,donor_3,stimulated,CD4 T cells,3147.0,0.0,125.0
3,AAACCGTGCTTCCG-1,MS4A1,0.0,donor_1,control,CD14+ Monocytes,2639.0,0.0,242.0
4,AAACCGTGCTTCCG-1,CD79A,0.0,donor_1,control,CD14+ Monocytes,2639.0,0.0,242.0


## Takeaway

`analysis` is a standard AnnData object with every aligned container intact,
ready for Scanpy or serialization. `plot_data` is an ordinary DataFrame with
exactly the values that were requested. The budget made that explicit before
the first matrix read.